# INSTALLING AND IMPORTING DEPENDENCIES

In [ ]:
pip install optuna

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import optuna
import pickle

# DATA LOADING

In [ ]:
df = pd.read_csv("../../data/car_auction_train.csv")

In [ ]:
df

In [ ]:
train, test = train_test_split(df, test_size=0.2)

# EXPLORATORY DATA ANALYSIS

I plot the correlation matrix of the numerical features. Also I visualize the selling price by car manufacturer, car counts, scatterplots, and NaN values. Key observations are:

1) The selling price is heavily skewed to the left with a few outliers stretching upto 250k USD. This shows that majority of the cars are sold a lower rate.

2) The sellingprice is highly correlated with condition and year of the manufacturing. This is obvious - as a car having better condition/recent year of manufacturing tends be highly attractive to buyers.

3) A few manufacturers such as Rolls-Royce, Ferrari, Lamborghini, Aston Martin hold their brand value and resell for much higher values. However being luxuries, they are also fewer in number.

4) Consumer car manufacturers such as Ford, Honda, Toyota, Chevrolet, Nissan dominate the share of market in numbers of cars sold.

5) Consumer car bodies such as sedan, wagon, hatchback, suv are more in number.

6) Most popular body and interior color are black, white, silver.

In [ ]:
train

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.isna().sum()

In [ ]:
numeric_cols = train.select_dtypes(include=np.number)
categorical_cols = train.select_dtypes(include="object")

In [ ]:
corr_map = numeric_cols.corr()
sns.heatmap(corr_map, annot=True)

In [ ]:
from pandas.plotting import scatter_matrix

scatter_matrix(numeric_cols,figsize=(12,8),diagonal="hist");

In [ ]:
sns.histplot(train["sellingprice"])

In [ ]:
sns.histplot(train["odometer"])

In [ ]:
price_by_make = train.groupby('make')["sellingprice"].mean()
plt.figure(figsize=(16,8))
plt.bar(price_by_make.index,price_by_make.values)
plt.xticks(rotation="vertical");

In [ ]:
plt.figure(figsize=(16,8))
plt.bar(train["make"].value_counts().index, np.log1p(train["make"].value_counts().values));
plt.xticks(rotation="vertical")
plt.xlabel("Car Manufacturer")
plt.ylabel("Selling Price (Logarithmic Scale)");

In [ ]:
top_body = train["body"].value_counts().sort_values(ascending=False).head(20)
plt.bar(top_body.index, top_body.values)
plt.xticks(rotation="vertical")
plt.xlabel("Car Segment")
plt.ylabel("Number of Cars");

In [ ]:
plt.bar(train["color"].value_counts().index, np.log1p(train["color"].value_counts().values))
plt.xticks(rotation="vertical")
plt.xlabel("Car Color")
plt.ylabel("Number of Cars (Logarithmic Scale)");

In [ ]:
plt.bar(train["interior"].value_counts().index, np.log1p(train["interior"].value_counts().values))
plt.xticks(rotation="vertical")
plt.xlabel("Car Interior")
plt.ylabel("Number of Cars (Logarithmic Scale)");

# DATA PREPROCESSING

1) **Drop Rows**:  
I dropped the rows where car company (make) and the model (model) were not present. This is one of the major indicators of the selling price comes from the car company and it's specific model. If we don't have information about this, then it is very difficult to predict the price.  

2) **Modify Names**:
I had various inconsistencies in the dataset of the car companies. Some 'make' where labeled in their uppercase, lowercase, first or abbreviated names (for ex: mercedez, mercedez-benz and Mercedez were all present). To resolve this, I created a cleanup map that replaces the different names into one standard naming system.

3) **Handling NaNs(categorical)**:  
    (i) I grouped the cars under their car company name and found the most frequent one they make for the 'trim' and 'body' columns. This was then imputed in the missing values of the respective columns, according to their respective groups.

    (ii) Columns 'transmission', 'color' and 'interior' were marked as 'Unknown' to let the model decide to associate them with any relations

4) **Handling NaNs(numerical)**:  
    (i) Odometer was imputed by its median.

    (ii)Condition was imputed based on the respective odometer reading. A car is usually in good condition if it has been driven less and vice versa so odometer reading gave fair insights about the condition.


In [ ]:
def dropRows(df):
  df = df[(df['make'].notna())]
  df = df[(df['model'].notna())]
  return df

In [ ]:
def modifyNames(df):
  df['make'] = df['make'].str.lower().str.strip()

  make_cleanup = {
    'chev truck': 'chevrolet',
    'gmc truck': 'gmc',
    'ford truck': 'ford',
    'dodge tk': 'dodge',
    'hyundai tk': 'hyundai',
    'mercedes-b': 'mercedes-benz',
    'mercedes': 'mercedes-benz',
    'landrover': 'land rover',
    'vw': 'volkswagen'
  }
  df['make'] = df['make'].replace(make_cleanup)

  df['make'] = df['make'].str.title()

  df['model'] = df['model'].str.lower().str.replace("-","").str.replace("/","")
  df['model'] = df['model'].str.title()

  return df

In [ ]:
def imputeUnknown(df):
  df['trim'] = df['trim'].fillna(df.groupby('make')['trim'].transform(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'))
  df['body'] = df['body'].fillna(df.groupby('make')['body'].transform(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'))

  df['transmission'] = df['transmission'].fillna("Unknown")
  df['color'] = df['color'].fillna("Unknown")
  df['interior'] = df['interior'].fillna("Unknown")

  return df

In [ ]:
def fillCondition(row):
  if row['odometer'] <= 30000:
    return 4.0
  elif (row['odometer'] > 30000) & (row['odometer'] <= 100000):
    return 3.0
  elif (row['odometer'] > 100000) & (row['odometer'] <= 200000):
    return 2.0
  else:
    return 1.0

In [ ]:
def imputeNumeric(df):
  df.loc[df['condition'].isna(),"condition"] = df.loc[df['condition'].isna()].apply(fillCondition, axis=1)

  odometer_median = df['odometer'].median()
  df['odometer'] = df['odometer'].fillna(odometer_median)
  return df

In [ ]:
def dataPreprocessing(df):
  df = dropRows(df)
  df = modifyNames(df)
  df = imputeUnknown(df)
  df = imputeNumeric(df)

  return df

In [ ]:
len(train["body"].unique())

# FEATURE ENGINEERING  

1) '**Years Used**' feature which shows how many years the car has been used - the higher the usage the lower the price (usually)

2) '**Miles Per Year**' feature which shows how many miles the car is driven on average in a year - indicative of its depreciation

3) '**Luxury Brands**' and '**Is Rare**' features which are indicative of some cars/companies that resell a lot. Also rare cars have high variance in their prices due to collectibles going after them so it is helpful to flag them

4) '**Depreciation**' feature which depicts how much the car's condition has worsened over the years of usage

In [ ]:
def featureEngineering(df):
  df['years_used'] = df['year'].apply(lambda x: 2016 - x)
  df["miles_per_year"] = df['odometer']/df['years_used']

  luxury_brands = ["Rolls-Royce","Ferrari","Lamborghini","Bentley","Porsche","Aston Martin"]
  df["luxury_brands"] = df['make'].apply(lambda x: 1 if x in luxury_brands else 0)
  rare_brands = ["Rolls-Royce","Lamborghini"]
  df["is_rare"] = df["make"].apply(lambda x: 1 if x in rare_brands else 0)
  df['depreciation'] = df['years_used']*(df['condition']/5)
  return df

In [ ]:
train = dataPreprocessing(train)
train = featureEngineering(train)

test = dataPreprocessing(test)
test = featureEngineering(test)

In [ ]:
train_target = train['sellingprice']
train = train.drop(columns=['sellingprice'])

test_target = test['sellingprice']
test = test.drop(columns=['sellingprice'])

# MODEL

My main model is lightgbm's **LGBMRegressor**. This is a **gradient boosting** algorithm that is based on the **decision tree** structure and grows **leaf-wise**. LightGBM is very efficient and fast and is highly capable of detecting non-linear relations especially in tabular datatset.

Before feeding into my model, I had to convert my categorical features into numeric ones as the regressor can't use string objects. This is done through **One-Hot Encoding** and **Target Encoding**. These encoders convert the string objects into numeric representation. These encoders are chained together in a **ColumnTransformer** which can be directly used in a **Pipeline**.

Initially, I tried to produce results in a **base LGBMRegressor** - no tuning was done. The results were impressive (**R2 score of 0.91 and RMSE of 2902.19**) showing the regressor's capabilities of extracting and learning useful relations within a **tabular dataset**.

In [ ]:
numeric_cols = train.select_dtypes(include=['int64','float64']).columns.to_list()
one_hot_cols = ['transmission']
target_cols = train.select_dtypes(include=['object']).columns.to_list()
target_cols = [obj for obj in target_cols if obj != 'transmission']

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("numeric","passthrough",numeric_cols),
    ("one_hot",OneHotEncoder(handle_unknown='ignore'),one_hot_cols),
    ("target",TargetEncoder(target_type='continuous'),target_cols)
])

In [ ]:
# BASE MODEL
model = LGBMRegressor(n_jobs=-1,verbose=-1)

pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",model)
])

pipeline.fit(train,train_target)
print(pipeline)
y_pred = pipeline.predict(test)

In [ ]:
r2 = r2_score(test_target,y_pred)
print(f"R2 Score: {r2}")
rmse = np.sqrt(mean_squared_error(test_target,y_pred))
print(f"RMSE Score: {rmse}")

# HYPERPARAMETER TUNING

The parameters for the regressor are tuned to obtain even better results. These parameters are as follows:

1) '**n_estimators**' which assigns the number of trees.
2) '**max_depth**' which assigns the max depth for each tree to grow.
3) '**num_leaves**' which assigns the number of leaves (leaf nodes) to grow in each tree.
4) '**min_child_samples**' which assigns the minimum number of samples in a leaf node.
5) '**learning rate**' which assigns the 'shrinkage factor' for each sequential tree added in the gradient boosting process

The hyperparameter tuning is done using a library called **Optuna**. This uses a **Bayesian** approach where a **probabilistic model** of the parameters is created using **past trial history**. So the **search space** is itself created into an objective which is then tried to **optimize**.

After tuning, I obtained the most promising parameters for my model which minimized my objective - RMSE. Using these parameters, the model was trained to obtain an **R2 Score: 0.95 and RMSE: 2244.34**, showing the effectiveness of the tuning process.

In [ ]:
# TUNED MODEL
def objective(trial):
  n_estimators = trial.suggest_int("n_estimators",100,3000,step=100)
  max_depth = trial.suggest_int("max_depth",2,10)
  num_leaves = trial.suggest_int("num_leaves",2,min(2**max_depth,256))
  learning_rate = trial.suggest_float("learning_rate",1e-3,1e-1,log=True)
  min_child_samples = trial.suggest_int("min_child_samples",10,100,step=10)

  model = LGBMRegressor(n_estimators=n_estimators,max_depth=max_depth, num_leaves=num_leaves,
                        learning_rate=learning_rate,min_child_samples=min_child_samples,n_jobs=-1)

  pipeline = Pipeline(steps=[
      ("preprocessor",preprocessor),
      ("model",model)
  ])

  score = cross_val_score(pipeline,train,train_target,cv=5,scoring="neg_root_mean_squared_error",n_jobs=-1)
  rmse = -score.mean()
  return rmse

In [ ]:
# study = optuna.create_study(direction="minimize")
# study.optimize(objective,n_trials=20)

In [ ]:
best_params = {
    "n_estimators":2400,
    "max_depth":10,
    "num_leaves":176,
    "learning_rate":0.03038452742864471,
    "min_child_samples":10,
}

model = LGBMRegressor(**best_params,n_jobs=-1,verbose=-1)

pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",model)
])

pipeline.fit(train,train_target)
print(pipeline)
y_pred = pipeline.predict(test)

In [ ]:
r2 = r2_score(test_target,y_pred)
print(f"R2 Score: {r2}")
rmse = np.sqrt(mean_squared_error(test_target,y_pred))
print(f"RMSE Score: {rmse}")

In [ ]:
model = pipeline.named_steps["model"]
preprocessor = pipeline.named_steps["preprocessor"]

importances = model.feature_importances_
feature_names =  preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({"Feature":feature_names,"Importance":importances}).sort_values(by="Importance",ascending=False)
importance_df

In [ ]:
plt.bar(importance_df["Feature"],importance_df["Importance"])
plt.xticks(rotation="vertical")
plt.xlabel("Features")
plt.ylabel("Feature Importance");

# QUANTILE MODEL

A secondary model is trained using **quantile regression**. Here instead of predicting the actual price, it predicts the **'alpha' percentile** of the price of similar commodities present together in its **leaf node**.

I used this quantile model to obtain the **10th percentile** price for each car. This is done so as to assess the **risk associated with each car**. If a car has higher difference in its predicted price and the 10th percentile price ($P_{average}$ - $P_{10}$), it is associated with **higher risk/volatility**.

For example consider a car with average predicted price as 15k.
If the car has a $P_{10}$ of 1k, then even in worse scenarios (bottom 10 percent of the prices) the car sells for 14k. However, if the $P_{10}$ value if 5k, then in worse scenarios the car sells for 10k. Thus the car is associated with more risk.

The quantile model acts as a measure of the risk and uncertainity which greatly helps in the bidding phase.

In [ ]:
# QUANTILE (10th Percentile) MODEL
quantile = LGBMRegressor(objective="quantile",alpha=0.1,metric="quantile",n_jobs=-1,verbose=-1,random_state=42)

quantile_pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("quantile",quantile)
])

quantile_pipeline.fit(train,train_target)
print(quantile_pipeline)
pred = quantile_pipeline.predict(test)

In [ ]:
# TRAINING ON FULL DATASET
final = pd.concat([train,test]).reset_index(drop=True)
final_target = pd.concat([train_target,test_target]).reset_index(drop=True)

In [ ]:
final_pred = pipeline.predict(final)
final_quantile_pred = quantile_pipeline.predict(final)

In [ ]:
r2 = r2_score(final_target,final_pred)
print(f"R2 Score: {r2}")
rmse = np.sqrt(mean_squared_error(final_target,final_pred))
print(f"RMSE Score: {rmse}")

# EXPORTING TO .pkl FILES

Three files are exported - the main model (which contains the regressor and the encoders together in a pipeline), the quantile model (which contains the 10th percentile predictor and the encoders together in a pipeline) and a dictionary map containing maps for data preprocessing (trim_map, body_map, cleanup_names and odometer_median).

In [ ]:
trim_map = final.groupby('make')['trim'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown").to_dict()
body_map = final.groupby('make')['body'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown").to_dict()

In [ ]:
cleanup_names = {
    'chev truck': 'chevrolet',
    'gmc truck': 'gmc',
    'ford truck': 'ford',
    'dodge tk': 'dodge',
    'hyundai tk': 'hyundai',
    'mercedes-b': 'mercedes-benz',
    'mercedes': 'mercedes-benz',
    'landrover': 'land rover',
    'vw': 'volkswagen'
}

In [ ]:
median = final['odometer'].median()

In [ ]:
others = {
    "cleanup_names":cleanup_names,
    "trim_map":trim_map,
    "body_map":body_map,
    "median":median
}

In [ ]:
pipeline_filename = "model.pkl"
with open(pipeline_filename,"wb") as f:
  pickle.dump(pipeline,f)

In [ ]:
quantile_filename = "quantile.pkl"
with open(quantile_filename,"wb") as f:
  pickle.dump(quantile_pipeline,f)

In [ ]:
others_filename = "others.pkl"
with open(others_filename,"wb") as f:
  pickle.dump(others,f)